# 27 — Atipicidad respecto de productos parecidos + umbral por segmento + precio exacto del vendedor

**Idea propia.** Un usado suele ser más barato, tener 1 unidad, menos fotos y no tener garantía, **comparado con el mismo producto nuevo**. El modelo ya compara contra la categoría (`category_id`, `c_cat_price`) y contra el vendedor, pero no contra **productos parecidos**. Mi EDA previo (versión burda: misma categoría + 3 primeras palabras del título, solo ARS): en vendedores sin historial, el 20% más barato es 29% `new` y el 20% más caro es 55% `new`; pero esa versión cubre solo el 3% de las filas.

**Partes:**
1. Vecinos por título (TF-IDF + coseno), dentro de la misma categoría y moneda, sin la propia fila ni las del mismo vendedor.
2. Features de "desvío respecto de los parecidos" (k = 10 y 30).
3. Medición sobre el HistGB P7a con duración y, si pasa, en el promedio final con las semillas 42 y 7. También la accuracy en vendedores sin historial.
4. Dos pruebas baratas: (a) umbral por segmento con las OOF del modelo final; (b) `TargetEncoder` de (vendedor, precio exacto) y (vendedor, categoría, precio exacto).

**Material** (relevado el 25/10; el 27/10 la terminal no tenía permiso para leer `material/`):
- vecinos más cercanos: kNN, `material/Knn/Presentacion_kNN.pdf` y `Copia de AP_kNN.ipynb`;
- similitud coseno: SIA, `S12 - Redes Neuronales - NLP.pdf`, slide 15;
- target encoder: `Copia de AP_Categóricas.ipynb`;
- umbral de decisión: SIA S8 y `Repaso de Metricas.pdf`;
- *feature engineering*: `Copia de AP_Categóricas.ipynb`.

**No está en el material:** TF-IDF y n-gramas de caracteres (ya explicados en `03_titulo` / `09_texto`).

In [1]:
import sys
sys.path.append("..")
import os, time, warnings
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from joblib import Memory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize, TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate

from features import (aplanar_anidadas, agregar_features_contenido, agregar_categoricas_crudas, agregar_combos,
                      PrediccionOOF, armar_modelo_texto, armar_pipeline_histgb_combos, armar_pipeline_final, INDICES_CAT)

pd.set_option("display.width", 200)
SEED = 42
crudo = pd.read_json("../data/train_data.jsonlines", lines=True)
crudo_test = pd.read_json("../data/test_data.jsonlines", lines=True)
y = (crudo["condition"] == "new").astype(int)
y_np = y.to_numpy()
n = len(crudo)
train = agregar_combos(agregar_categoricas_crudas(agregar_features_contenido(aplanar_anidadas(crudo))))
folds = np.empty(n, dtype=int)
for k, (_, v) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(train, y)):
    folds[v] = k + 1
carpeta_lab = ".cache_laboratorio"
memoria_texto = Memory(location=carpeta_lab, verbose=0)

oof_ref = pd.read_csv(os.path.join(carpeta_lab, "oof_p7c_duracion.csv"))
assert (oof_ref["fold"].to_numpy() == folds).all()
p_hgb, p_cat = oof_ref["histgb"].to_numpy(), oof_ref["catboost"].to_numpy()
p_final = 0.5 * p_hgb + 0.5 * p_cat


def acc_por_fold(prob, filas=None, umbral=0.5):
    filas = np.ones(n, bool) if filas is None else filas
    acierto = ((np.asarray(prob) >= umbral).astype(int) == y_np)
    return pd.Series(acierto[filas]).groupby(folds[filas]).mean().to_numpy()


def comparar(acc, ref):
    diferencias = np.round(acc - ref, 10)
    gana = int((diferencias > 0).sum())
    return {"media": acc.mean(), "dif_media": diferencias.mean(), "dif_desvio": diferencias.std(), "folds_gana": gana,
            "pasa": gana == 5 or (gana == 4 and diferencias.mean() > diferencias.std())}


# Vendedores "sin historial" en cada fold: no tienen ninguna publicación en el train de ese fold
sin_historial = np.zeros(n, bool)
for k in range(1, 6):
    va = folds == k
    sin_historial[va] = ~train.loc[va, "seller_id"].isin(set(train.loc[~va, "seller_id"])).to_numpy()
acc_hgb, acc_final = acc_por_fold(p_hgb), acc_por_fold(p_final)
print("HistGB P7a + duración:", acc_hgb.mean().round(4), "| sin historial:", acc_por_fold(p_hgb, sin_historial).mean().round(4),
      f"({sin_historial.mean():.1%} de las filas)")
print("Modelo final:         ", acc_final.mean().round(4), "| sin historial:", acc_por_fold(p_final, sin_historial).mean().round(4))

HistGB P7a + duración: 0.9412 | sin historial: 0.8923 (30.7% de las filas)
Modelo final:          0.942 | sin historial: 0.8931


## 1. Vecinos por título

**Qué hace:**
1. TF-IDF de los títulos de **train + test** (palabras 1-2 + caracteres 3-5, como el modelo de texto), normalizado: así el producto escalar entre dos filas **es** la similitud coseno (S12, slide 15).
2. Para cada grupo (`category_id`, moneda) se calculan las similitudes de todos contra todos.
3. Se descartan la **propia fila** y las del **mismo vendedor** (se quiere medir el mercado, no al vendedor, que ya está en el modelo).
4. Se guardan los 30 más parecidos de cada publicación.

Es la idea de kNN (`Presentacion_kNN.pdf`): la "respuesta" sale de los vecinos más cercanos. Pero acá **no se promedia el target** de los vecinos (eso ya se probó en I1 y no sumó), sino sus **features**, para medir cuánto se aparta la publicación de su mercado.

**Por qué no es leakage:** el TF-IDF y los vecinos se calculan solo con texto y features que existen en el test; **no se usa `condition`**. Una fila de validación y una de test buscan vecinos en el mismo universo (train + test), así que se describen igual. No hay etiquetas que filtrar, así que no hace falta *cross-fitting* (a diferencia de `PrediccionOOF` o del `TargetEncoder`).

**Limitación (como R0a y R3):** es transductivo. En producción, los vecinos se buscarían contra el histórico disponible.

In [2]:
universo = aplanar_anidadas(pd.concat([crudo.drop(columns="condition"), crudo_test], ignore_index=True))
titulos = universo["title"].fillna("")
inicio = time.time()
X_pal = TfidfVectorizer(ngram_range=(1, 2), min_df=2, strip_accents="unicode").fit_transform(titulos)
X_car = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, strip_accents="unicode").fit_transform(titulos)
X = normalize(hstack([X_pal, X_car]).tocsr())

K = 30
vec_idx = np.full((len(universo), K), -1, dtype=np.int64)
vec_sim = np.full((len(universo), K), np.nan, dtype=np.float32)
vendedor = universo["seller_id"].to_numpy()
for (_, _), filas in universo.groupby(["category_id", "currency_id"]).indices.items():
    if len(filas) < 2:
        continue
    S = (X[filas] @ X[filas].T).toarray().astype(np.float32)
    S[vendedor[filas][:, None] == vendedor[filas][None, :]] = -1     # propia fila y mismo vendedor
    m = min(K, len(filas) - 1)
    top = np.argpartition(-S, m - 1, axis=1)[:, :m] if m < len(filas) else np.argsort(-S, axis=1)[:, :m]
    sims = np.take_along_axis(S, top, axis=1)
    orden = np.argsort(-sims, axis=1)
    top, sims = np.take_along_axis(top, orden, axis=1), np.take_along_axis(sims, orden, axis=1)
    sims[sims < 0] = np.nan                                           # vecinos descartados (mismo vendedor)
    vec_idx[filas, :m] = np.where(np.isnan(sims), -1, filas[top])
    vec_sim[filas, :m] = sims
print(f"Vecinos calculados en {(time.time() - inicio) / 60:.1f} min")
print("Similitud del vecino más parecido (cuantiles):", np.round(np.nanquantile(vec_sim[:, 0], [0.1, 0.25, 0.5, 0.75, 0.9]), 3))
print("% de filas sin ningún vecino (categoría-moneda con un solo vendedor):", np.mean(np.isnan(vec_sim[:, 0])).round(4))

Vecinos calculados en 0.1 min
Similitud del vecino más parecido (cuantiles): [0.07  0.16  0.274 0.422 0.575]
% de filas sin ningún vecino (categoría-moneda con un solo vendedor): 0.0431


## 2. Features de "desvío respecto de los parecidos"

Un vecino es **válido** si su similitud es ≥ 0,3 (fijado a priori: con 0,3 dos títulos comparten palabras/fragmentos importantes; no se eligió mirando el target). Si hay menos de 3 vecinos válidos, las features son NaN (el HistGB maneja NaN).

Para k = 10 y k = 30:
- `precio_rel`: log(1 + precio) − mediana de los vecinos; `precio_pct`: fracción de vecinos más baratos (0 = el más barato, 1 = el más caro);
- `cant_rel`: log(1 + initial_quantity) − mediana de los vecinos;
- `fotos_rel`: n_fotos − mediana de los vecinos; `garantia_rel`: tiene_garantia − media de los vecinos;
- `mismo_tipo`: % de vecinos con el mismo `listing_type_id`;
- `sim_media`: similitud media con los vecinos válidos (qué tan "estándar" es el producto); `n_validos`.

El precio es el **crudo** (sin capear), pero la mediana de los vecinos es robusta a los precios mal cargados.

In [3]:
UMBRAL_SIM, MIN_VALIDOS = 0.3, 3
valores = {
    "precio": np.log1p(universo["price"].clip(lower=0)).to_numpy(),
    "cant": np.log1p(universo["initial_quantity"]).to_numpy(),
    "fotos": universo["n_fotos"].to_numpy(dtype=float),
    "garantia": universo["tiene_garantia"].to_numpy(dtype=float),
}
tipo = universo["listing_type_id"].to_numpy()


def features_atipicidad(k):
    idx, sim = vec_idx[:, :k], vec_sim[:, :k]
    valido = (idx >= 0) & (sim >= UMBRAL_SIM)
    n_val = valido.sum(axis=1)
    hay = n_val >= MIN_VALIDOS
    idx0 = np.where(valido, idx, 0)
    f = {}
    for nombre, v in valores.items():
        vec = np.where(valido, v[idx0], np.nan)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ref = np.nanmean(vec, axis=1) if nombre == "garantia" else np.nanmedian(vec, axis=1)
        f[f"{nombre}_rel"] = np.where(hay, v - ref, np.nan)
        if nombre == "precio":
            menor = np.where(valido, (v[idx0] < v[:, None]) + 0.5 * (v[idx0] == v[:, None]), 0).sum(axis=1)
            f["precio_pct"] = np.where(hay, menor / np.maximum(n_val, 1), np.nan)
    f["mismo_tipo"] = np.where(hay, np.where(valido, tipo[idx0] == tipo[:, None], 0).sum(axis=1) / np.maximum(n_val, 1), np.nan)
    f["sim_media"] = np.where(hay, np.nanmean(np.where(valido, sim, np.nan), axis=1), np.nan)
    f["n_validos"] = n_val.astype(float)
    return pd.DataFrame({f"at{k}_{c}": x for c, x in f.items()})


atip = pd.concat([features_atipicidad(10), features_atipicidad(30)], axis=1)
for c in atip:
    train[c] = atip[c].to_numpy()[:n]
cols10, cols30 = [c for c in atip if c.startswith("at10_")], [c for c in atip if c.startswith("at30_")]
es_test = np.arange(len(universo)) >= n
print("% de filas con features (≥ 3 vecinos válidos): k=10 train",
      atip["at10_precio_rel"][~es_test].notna().mean().round(3), "test", atip["at10_precio_rel"][es_test].notna().mean().round(3),
      "| k=30 train", atip["at30_precio_rel"][~es_test].notna().mean().round(3))

/var/folders/8f/fp5mmq2j48sc0h6x9_4cw75r0000gn/T/ipykernel_27778/3426097168.py:28: RuntimeWarning: Mean of empty slice
  f["sim_media"] = np.where(hay, np.nanmean(np.where(valido, sim, np.nan), axis=1), np.nan)


% de filas con features (≥ 3 vecinos válidos): k=10 train 0.184 test 0.185 | k=30 train 0.184


/var/folders/8f/fp5mmq2j48sc0h6x9_4cw75r0000gn/T/ipykernel_27778/3426097168.py:28: RuntimeWarning: Mean of empty slice
  f["sim_media"] = np.where(hay, np.nanmean(np.where(valido, sim, np.nan), axis=1), np.nan)


In [4]:
# EDA: % de new por quintil de precio relativo (k = 30), en todos y en vendedores con 1 sola publicación en train
quintil = pd.qcut(train["at30_precio_pct"], 5, labels=["0-20% (más barato)", "20-40", "40-60", "60-80", "80-100% (más caro)"], duplicates="drop")
un_vendedor = train["seller_id"].map(train["seller_id"].value_counts()).to_numpy() == 1
print(pd.DataFrame({"filas": quintil.value_counts().sort_index(), "pct_new": y.groupby(quintil, observed=False).mean(),
                    "pct_new (vendedores con 1 publicación)": y[un_vendedor].groupby(quintil[un_vendedor], observed=False).mean()}).round(3))
for c in ["at30_cant_rel", "at30_fotos_rel", "at30_garantia_rel", "at30_mismo_tipo"]:
    tramo = pd.cut(train[c], [-np.inf, -0.01, 0.01, np.inf], labels=["menos que los parecidos", "igual", "más que los parecidos"])
    print(f"\n{c}:", y.groupby(tramo, observed=False).agg(["size", "mean"]).round(3).to_dict(orient="index"))

                    filas  pct_new  pct_new (vendedores con 1 publicación)
at30_precio_pct                                                           
0-20% (más barato)   2715    0.518                                   0.385
20-40                2520    0.576                                   0.420
40-60                3437    0.602                                   0.477
60-80                1654    0.641                                   0.489
80-100% (más caro)   2565    0.647                                   0.522

at30_cant_rel: {'menos que los parecidos': {'size': 2257, 'mean': 0.83}, 'igual': {'size': 7685, 'mean': 0.386}, 'más que los parecidos': {'size': 2949, 'mean': 0.952}}

at30_fotos_rel: {'menos que los parecidos': {'size': 4590, 'mean': 0.586}, 'igual': {'size': 3320, 'mean': 0.647}, 'más que los parecidos': {'size': 4981, 'mean': 0.564}}

at30_garantia_rel: {'menos que los parecidos': {'size': 5634, 'mean': 0.534}, 'igual': {'size': 3229, 'mean': 0.491}, 'más que los p

## 3. Medición sobre el HistGB P7a con duración

Cada configuración se suma al HistGB P7a + duración (`armar_pipeline_histgb_combos()`, modelo de texto cacheado) y se compara fold por fold contra 0,9412. Se usan probabilidades *out-of-fold* (`cross_val_predict`) para poder medir también la accuracy en **vendedores sin historial** (los que no tienen ninguna publicación en el train del fold).

Primero las dos familias completas (k = 10 y k = 30); después cada feature de la mejor k por separado.

In [5]:
def con_texto_cacheado(pipeline):
    transformers = pipeline.named_steps["preprocesador"].transformers
    i = [nm for nm, _, _ in transformers].index("texto")
    transformers[i] = ("texto", Pipeline([("oof", PrediccionOOF(armar_modelo_texto(), random_state=SEED)),
                                          ("identidad", "passthrough")], memory=memoria_texto), ["title", "warranty_texto"])
    return pipeline


def p7a_con(extras):
    pipeline = con_texto_cacheado(armar_pipeline_histgb_combos())
    pipeline.named_steps["preprocesador"].transformers.extend(extras)
    return pipeline


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
resultados, probs = {}, {}


def medir(nombre, extras, sola_cols=None):
    inicio = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        prob = cross_val_predict(p7a_con(extras), train, y, cv=cv, method="predict_proba", n_jobs=5)[:, 1]
        sola = (cross_validate(HistGradientBoostingClassifier(learning_rate=0.1, max_depth=None, max_iter=200, random_state=SEED),
                               train[sola_cols], y, cv=cv, scoring="accuracy", n_jobs=5)["test_score"].mean() if sola_cols else np.nan)
    probs[nombre] = prob
    a_sh, r_sh = acc_por_fold(prob, sin_historial), acc_por_fold(p_hgb, sin_historial)
    resultados[nombre] = {"señal_sola": sola, **comparar(acc_por_fold(prob), acc_hgb),
                          "sin_hist_antes": r_sh.mean(), "sin_hist_despues": a_sh.mean(),
                          "sin_hist_folds_gana": int((np.round(a_sh - r_sh, 10) > 0).sum()), "minutos": (time.time() - inicio) / 60}
    r = resultados[nombre]
    print(f"{nombre:34s} {r['media']:.4f} dif {r['dif_media']:+.4f} ({r['folds_gana']}/5){' PASA' if r['pasa'] else ''} | "
          f"sin historial {r['sin_hist_antes']:.4f} -> {r['sin_hist_despues']:.4f} | {r['minutos']:.1f} min")


medir("k=10 (8 features)", [("at", "passthrough", cols10)], cols10)
medir("k=30 (8 features)", [("at", "passthrough", cols30)], cols30)
mejor_k = "at10_" if resultados["k=10 (8 features)"]["dif_media"] >= resultados["k=30 (8 features)"]["dif_media"] else "at30_"
for c in [c for c in atip if c.startswith(mejor_k)]:
    medir(c, [("at", "passthrough", [c])], [c])
pd.DataFrame(resultados).T.round(4)

k=10 (8 features)                  0.9409 dif -0.0003 (1/5) | sin historial 0.8923 -> 0.8914 | 0.6 min


k=30 (8 features)                  0.9410 dif -0.0003 (2/5) | sin historial 0.8923 -> 0.8910 | 0.6 min


at30_precio_rel                    0.9407 dif -0.0006 (2/5) | sin historial 0.8923 -> 0.8908 | 0.5 min


at30_precio_pct                    0.9411 dif -0.0001 (2/5) | sin historial 0.8923 -> 0.8916 | 0.5 min


at30_cant_rel                      0.9409 dif -0.0003 (1/5) | sin historial 0.8923 -> 0.8915 | 0.6 min


at30_fotos_rel                     0.9410 dif -0.0002 (1/5) | sin historial 0.8923 -> 0.8921 | 0.5 min


at30_garantia_rel                  0.9407 dif -0.0006 (0/5) | sin historial 0.8923 -> 0.8902 | 0.5 min


at30_mismo_tipo                    0.9411 dif -0.0001 (2/5) | sin historial 0.8923 -> 0.8916 | 0.6 min


at30_sim_media                     0.9408 dif -0.0004 (2/5) | sin historial 0.8923 -> 0.8909 | 0.6 min


at30_n_validos                     0.9409 dif -0.0003 (1/5) | sin historial 0.8923 -> 0.8904 | 0.5 min


,señal_sola,media,dif_media,dif_desvio,folds_gana,pasa,sin_hist_antes,sin_hist_despues,sin_hist_folds_gana,minutos
k=10 (8 features),0.571514,0.9409,-0.000343,0.000683,1,False,0.892298,0.891366,0,0.600537
k=30 (8 features),0.572057,0.940986,-0.000257,0.000722,2,False,0.892298,0.890988,1,0.596874
at30_precio_rel,0.537814,0.940671,-0.000571,0.001147,2,False,0.892298,0.890792,1,0.545142
at30_precio_pct,0.537071,0.941114,-0.000129,0.000736,2,False,0.892298,0.891595,3,0.530815
at30_cant_rel,0.561814,0.940943,-0.0003,0.000481,1,False,0.892298,0.891469,1,0.561966
at30_fotos_rel,0.536929,0.941029,-0.000214,0.000559,1,False,0.892298,0.892058,1,0.548321
at30_garantia_rel,0.540271,0.940671,-0.000571,0.000282,0,False,0.892298,0.89016,0,0.538254
at30_mismo_tipo,0.537686,0.941143,-0.0001,0.000416,2,False,0.892298,0.891559,1,0.553087
at30_sim_media,0.535757,0.940843,-0.0004,0.000708,2,False,0.892298,0.890894,0,0.557788
at30_n_validos,0.536529,0.940929,-0.000314,0.000554,1,False,0.892298,0.890402,0,0.532644


## 4a. Umbral distinto por segmento (con las OOF guardadas del modelo final)

**Qué hace.** El modelo final predice `new` si P(new) ≥ 0,5. En la Fase C (P5) se vio que, para **todo** el conjunto, 0,5 ya es óptimo. Pero puede que en un segmento puntual el modelo esté descalibrado. Por ejemplo, en los vendedores sin historial podría convenir exigir más (o menos) para decir `new`. Material: umbral de decisión, SIA S8 y `Repaso de Metricas.pdf` (curva ROC).

Segmentos: (1) vendedores **sin historial** en el fold; (2) `listing_type_id = free`; (3) los dos a la vez (umbral propio para cada uno y 0,5 para el resto).

**Sin trampa:** el umbral de cada segmento se elige en una grilla de 0,30 a 0,70 (paso 0,01) **con 4 folds** y se evalúa en el **quinto**, rotando. No se reentrena nada: se usan las probabilidades OOF del modelo final de la 18.

In [6]:
grilla = np.round(np.arange(0.30, 0.701, 0.01), 2)
free = train["listing_type_id"].to_numpy() == "free"


def umbral_por_segmento(segmentos):
    pred = (p_final >= 0.5).astype(int)
    elegidos = {}
    for k in range(1, 6):
        entrena, evalua = folds != k, folds == k
        for nombre, seg in segmentos.items():
            filas_e = entrena & seg
            mejor = max(grilla, key=lambda u: (((p_final[filas_e] >= u).astype(int)) == y_np[filas_e]).mean())
            elegidos.setdefault(nombre, []).append(mejor)
            filas_v = evalua & seg
            pred[filas_v] = (p_final[filas_v] >= mejor).astype(int)
    acc = pd.Series(pred == y_np).groupby(folds).mean().to_numpy()
    return acc, elegidos


for nombre, segmentos in {"umbral: sin historial": {"sin historial": sin_historial},
                          "umbral: free": {"free": free & ~sin_historial},
                          "umbral: sin historial + free": {"sin historial": sin_historial, "free": free & ~sin_historial}}.items():
    acc, elegidos = umbral_por_segmento(segmentos)
    resultados[nombre + " (vs modelo final)"] = comparar(acc, acc_final)
    print(nombre, "| umbrales elegidos por fold:", elegidos, "|", {k: round(float(v), 4) for k, v in comparar(acc, acc_final).items()})

umbral: sin historial | umbrales elegidos por fold: {'sin historial': [np.float64(0.49), np.float64(0.48), np.float64(0.49), np.float64(0.49), np.float64(0.49)]} | {'media': 0.9421, 'dif_media': 0.0001, 'dif_desvio': 0.0001, 'folds_gana': 3.0, 'pasa': 0.0}
umbral: free | umbrales elegidos por fold: {'free': [np.float64(0.54), np.float64(0.54), np.float64(0.57), np.float64(0.57), np.float64(0.57)]} | {'media': 0.942, 'dif_media': 0.0, 'dif_desvio': 0.0, 'folds_gana': 1.0, 'pasa': 0.0}


umbral: sin historial + free | umbrales elegidos por fold: {'sin historial': [np.float64(0.49), np.float64(0.48), np.float64(0.49), np.float64(0.49), np.float64(0.49)], 'free': [np.float64(0.54), np.float64(0.54), np.float64(0.57), np.float64(0.57), np.float64(0.57)]} | {'media': 0.9421, 'dif_media': 0.0001, 'dif_desvio': 0.0001, 'folds_gana': 4.0, 'pasa': 1.0}


## 4b. `TargetEncoder` de (vendedor, precio exacto) y (vendedor, categoría, precio exacto)

**Qué hace.** `c_seller_price` ya combina vendedor × **rango** de precio. La idea es usar el precio **exacto**: un vendedor que publica muchas veces el mismo artículo al mismo precio (stock de tienda) contra uno que pone precios distintos. Se crean las combinaciones como texto (determinístico) y se codifican con `TargetEncoder` (*cross-fitting* + *smoothing*, `Copia de AP_Categóricas.ipynb` para el target encoder), **adentro del `Pipeline`**: se aprende solo con el train de cada fold, así que no es leakage. Se usa el precio crudo.

In [7]:
train["c_seller_precio_exacto"] = train["seller_id"].astype(str) + "_" + crudo["price"].astype(str)
train["c_seller_cat_precio_exacto"] = train["seller_id"].astype(str) + "_" + train["category_id"].astype(str) + "_" + crudo["price"].astype(str)
for nombre, cols in {"TE (vendedor, precio exacto)": ["c_seller_precio_exacto"],
                     "TE (vendedor, categoría, precio exacto)": ["c_seller_cat_precio_exacto"],
                     "TE ambos": ["c_seller_precio_exacto", "c_seller_cat_precio_exacto"]}.items():
    medir(nombre, [("te_precio", TargetEncoder(target_type="binary", random_state=SEED), cols)])
tabla = pd.DataFrame(resultados).T
tabla.to_csv(os.path.join(carpeta_lab, "atipicidad.csv"))
tabla.round(4)

TE (vendedor, precio exacto)       0.9409 dif -0.0003 (2/5) | sin historial 0.8923 -> 0.8917 | 0.6 min


TE (vendedor, categoría, precio exacto) 0.9412 dif -0.0001 (1/5) | sin historial 0.8923 -> 0.8909 | 0.6 min


TE ambos                           0.9409 dif -0.0004 (1/5) | sin historial 0.8923 -> 0.8911 | 0.5 min


,señal_sola,media,dif_media,dif_desvio,folds_gana,pasa,sin_hist_antes,sin_hist_despues,sin_hist_folds_gana,minutos
k=10 (8 features),0.571514,0.9409,-0.000343,0.000683,1,False,0.892298,0.891366,0,0.600537
k=30 (8 features),0.572057,0.940986,-0.000257,0.000722,2,False,0.892298,0.890988,1,0.596874
at30_precio_rel,0.537814,0.940671,-0.000571,0.001147,2,False,0.892298,0.890792,1,0.545142
at30_precio_pct,0.537071,0.941114,-0.000129,0.000736,2,False,0.892298,0.891595,3,0.530815
at30_cant_rel,0.561814,0.940943,-0.0003,0.000481,1,False,0.892298,0.891469,1,0.561966
at30_fotos_rel,0.536929,0.941029,-0.000214,0.000559,1,False,0.892298,0.892058,1,0.548321
at30_garantia_rel,0.540271,0.940671,-0.000571,0.000282,0,False,0.892298,0.89016,0,0.538254
at30_mismo_tipo,0.537686,0.941143,-0.0001,0.000416,2,False,0.892298,0.891559,1,0.553087
at30_sim_media,0.535757,0.940843,-0.0004,0.000708,2,False,0.892298,0.890894,0,0.557788
at30_n_validos,0.536529,0.940929,-0.000314,0.000554,1,False,0.892298,0.890402,0,0.532644


## Robustez del umbral por segmento: semilla 7

El umbral "sin historial + free" pasó con la semilla 42 (+0,0001, 4/5). Se repite **todo** con otros folds (`random_state=7`): se recalculan las OOF del modelo final (CatBoost + HistGB con duración) con esos folds, se redefine "sin historial" para esos folds y se eligen los umbrales con 4 folds y se evalúan en el quinto. Referencia: el modelo final con umbral 0,5 en los mismos folds (0,9422).

In [8]:
archivo_s7 = os.path.join(carpeta_lab, "oof_final_semilla7.csv")
cv7 = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
if os.path.exists(archivo_s7):
    oof7 = pd.read_csv(archivo_s7)
else:
    inicio = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        p7_cat = cross_val_predict(con_texto_cacheado(armar_pipeline_final()), train, y, cv=cv7, method="predict_proba", n_jobs=5,
                                   params={"modelo__cat_features": INDICES_CAT})[:, 1]
        p7_hgb = cross_val_predict(con_texto_cacheado(armar_pipeline_histgb_combos()), train, y, cv=cv7, method="predict_proba", n_jobs=5)[:, 1]
    folds7 = np.empty(n, dtype=int)
    for k, (_, v) in enumerate(cv7.split(train, y)):
        folds7[v] = k + 1
    oof7 = pd.DataFrame({"fold": folds7, "y": y_np, "catboost": p7_cat, "histgb": p7_hgb})
    oof7.to_csv(archivo_s7, index=False)
    print(f"OOF semilla 7: {(time.time() - inicio) / 60:.1f} min")

f7, pf7 = oof7["fold"].to_numpy(), 0.5 * oof7["catboost"].to_numpy() + 0.5 * oof7["histgb"].to_numpy()
sh7 = np.zeros(n, bool)
for k in range(1, 6):
    va = f7 == k
    sh7[va] = ~train.loc[va, "seller_id"].isin(set(train.loc[~va, "seller_id"])).to_numpy()
acc7_base = pd.Series((pf7 >= 0.5).astype(int) == y_np).groupby(f7).mean().to_numpy()
pred = (pf7 >= 0.5).astype(int)
elegidos7 = {}
for k in range(1, 6):
    entrena, evalua = f7 != k, f7 == k
    for nombre, seg in {"sin historial": sh7, "free": free & ~sh7}.items():
        filas_e = entrena & seg
        mejor = max(grilla, key=lambda u: (((pf7[filas_e] >= u).astype(int)) == y_np[filas_e]).mean())
        elegidos7.setdefault(nombre, []).append(mejor)
        pred[evalua & seg] = (pf7[evalua & seg] >= mejor).astype(int)
acc7 = pd.Series(pred == y_np).groupby(f7).mean().to_numpy()
print("Modelo final semilla 7 (umbral 0,5):", acc7_base.round(4), acc7_base.mean().round(4))
print("Umbrales elegidos:", elegidos7)
print("Predicciones que cambian:", int(((pf7 >= 0.5).astype(int) != pred).sum()), "de", n)
resultados["umbral: sin historial + free, SEMILLA 7 (vs modelo final s7)"] = comparar(acc7, acc7_base)
print({k: round(float(v), 5) for k, v in comparar(acc7, acc7_base).items()})

OOF semilla 7: 2.4 min
Modelo final semilla 7 (umbral 0,5): [0.9446 0.9426 0.9424 0.9401 0.9411] 0.9422
Umbrales elegidos: {'sin historial': [np.float64(0.51), np.float64(0.5), np.float64(0.51), np.float64(0.5), np.float64(0.51)], 'free': [np.float64(0.66), np.float64(0.58), np.float64(0.58), np.float64(0.7), np.float64(0.7)]}
Predicciones que cambian: 34 de 70000
{'media': 0.94213, 'dif_media': -3e-05, 'dif_desvio': 6e-05, 'folds_gana': 0.0, 'pasa': 0.0}


## Conclusión — nada pasa

| Prueba | Contra | Diferencia (folds) | Vendedores sin historial |
|---|---|---|---|
| Atipicidad k = 10 (8 features) | HistGB P7a + duración (0,9412) | −0,0003 (1/5) | 0,8923 → 0,8914 |
| Atipicidad k = 30 (8 features) | ídem | −0,0003 (2/5) | 0,8923 → 0,8910 |
| Cada feature de k = 30 por separado | ídem | −0,0001 a −0,0006 (0-2/5) | todas bajan o quedan igual |
| Umbral por segmento: sin historial | modelo final (0,9420) | +0,00007 (3/5) | — |
| Umbral por segmento: `free` | ídem | +0,00001 (1/5) | — |
| Umbral por segmento: los dos, **semilla 42** | ídem | +0,00009 (4/5, pasa por el criterio) | — |
| Umbral por segmento: los dos, **semilla 7** | modelo final s7 (0,9422) | **−0,00003 (0/5)** → no se confirma | — |
| `TargetEncoder` (vendedor, precio exacto) / (vendedor, categoría, precio exacto) / ambos | HistGB P7a + duración | −0,0003 / −0,0001 / −0,0004 | bajan |

- **Atipicidad:**
  - Cobertura baja: solo el **18,4%** de las filas tiene ≥ 3 vecinos con similitud ≥ 0,3. La mediana de la similitud con el vecino más parecido es 0,27: dentro de una categoría, los títulos de **otros** vendedores suelen ser productos distintos.
  - Donde hay vecinos, la EDA va en el sentido esperado: precio del quintil más barato al más caro, 38,5% → 52,2% `new` en vendedores con 1 publicación; cantidad distinta de los parecidos, 83-95% `new` contra 39% igual; más garantía que los parecidos, 76% contra 49-53%.
  - **No suma:** lo que dicen esas comparaciones ya lo dicen `initial_quantity`, `tiene_garantia`, el precio + `c_cat_price` y el texto.
- **Umbral por segmento:** la ganancia de la semilla 42 son ~6 predicciones de 70.000, y con la semilla 7 desaparece. Los umbrales elegidos cambian entre semillas (sin historial 0,48-0,49 → 0,50-0,51; `free` 0,54-0,57 → 0,58-0,70): el modelo está bien calibrado también por segmento (igual que P5 para el total).
- **`TargetEncoder` del precio exacto:** no suma sobre `c_seller_price` (vendedor × rango de precio) y el resto.
- **Decisión:** sin cambios y sin submission. Modelo final = P7c + duración (0,9420). Resultados en `.cache_laboratorio/atipicidad.csv`; OOF del modelo final con la semilla 7 en `oof_final_semilla7.csv` (sirve para futuras confirmaciones baratas).